# Generate Sentence completions for RealToxicityPrompts dataset

In [1]:
from pathlib import Path
from typing import Optional, List, Iterable, Dict, Any

import pandas as pd
import torch
from tqdm import tqdm
from transformers import GPT2Tokenizer, GPT2Model
from transformers import GPT2LMHeadModel
import json
from utils.utils import load_cache

# Load dataset and generate responses

In [ ]:
dataset_file = "../Datasets/RealToxicityPrompts-Dataset/prompts.jsonl"
output_dir = "../Datasets/RealToxicityPrompts-Output/gpt2/"

def gpt2(prompts: pd.Series,
         max_len: int,
         num_samples: int,
         model_name_or_path: str,
         out_file: Path,
         seed: int = 42,
         temperature: float = 1.0,
         top_k: int = 0,
         top_p: float = 0.9,
         **generate_kwargs) -> Iterable[str]:
    # Set up device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu = torch.cuda.device_count()
    torch.manual_seed(seed)
    if n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

    # Setup model
    tokenizer = GPT2Tokenizer.from_pretrained(model_name_or_path)
    model = GPT2LMHeadModel.from_pretrained(model_name_or_path).to(device)
    model.eval()

    # Load cached generations
    num_cached_generations = 0
    for generation in load_cache(out_file):
        yield generation
        num_cached_generations += 1

    # Remove prompts that have already been generated with
    prompts = prompts[num_cached_generations:]
    if prompts.empty:
        return

    # Generate with prompts
    for prompt in tqdm(prompts, desc='Generation', dynamic_ncols=True):
        encoded_input = tokenizer(prompt, return_tensors='pt').to(device)
        output_ids = model.generate(
            **encoded_input,
            max_new_tokens=max_len,
            num_return_sequences=num_samples,  # Generate 1 sample per prompt
            do_sample=True,
            top_k=top_k,
            top_p=top_p,
            temperature=temperature,
            **generate_kwargs
        )

        generation = tokenizer.decode(output_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        with out_file.open('a') as f:
            print(json.dumps(generation), file=f)
        yield generation

# Load dataset and generate responses
dataset = pd.read_json(dataset_file, lines=True)
prompts = pd.json_normalize(dataset['prompt'])['text']

output_dir = Path(output_dir)
output_dir.mkdir(exist_ok=True)
generations_file = output_dir / 'generations.jsonl'

generations_iter = gpt2(prompts=prompts,
                        max_len=20,
                        num_samples=1,
                        model_name_or_path='openai-community/gpt2',
                        out_file=generations_file,
                        seed=42,
                        temperature=1.0,
                        top_k=0,
                        top_p=0.9)

for _ in generations_iter:
    pass

print("Generations saved to generations.jsonl")
torch.cuda.empty_cache()

Generation:   0%|          | 30/99442 [00:06<5:23:16,  5.13it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Generation:   0%|          | 97/99442 [00:19<5:22:54,  5.13it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Generation:   0%|          | 120/99442 [00:23<5:22:06,  5.14it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
